## Normalizing human charts

This notebook documents the process of normalizing human-authored DDR step charts from the Dance Dance Convolution (DDC) dataset format into a simplified JSON structure.

### Source data
- **Input**: `data/unprocessed/json_filt/{fraxtil,itg}/**/*.json`
- **Output**: `data/real/{fraxtil,itg}/*_human.json`

### Format fransformation

#### Input format
The DDC dataset stores notes as arrays with the following structure:
```
[[measure, subdivision, offset], beat, time_seconds, step]
```

#### Output format
We simplify this to:
```json
{
  "dataset": "fraxtil",
  "song_name": "Sleaze",
  "charts": {
    "Challenge": {
      "difficulty": "Challenge",
      "steps": [{"time": 8.888, "step": "0101"}, ...],
      "num_steps": 500
    }
  }
}
```


In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
UNPROCESSED_DIR = PROJECT_ROOT / 'data' / 'unprocessed' / 'json_filt'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'real'
GROUPS = ['fraxtil', 'itg']
ALLOWED_DIFFICULTIES = {'Beginner', 'Easy', 'Medium', 'Hard', 'Challenge'}

In [2]:
def check_already_processed():
    existing = list(OUTPUT_DIR.glob('**/*_human.json'))
    if existing:
        sample = existing[0]
        with open(sample, 'r') as f:
            data = json.load(f)
        first_chart = next(iter(data.get('charts', {}).values()), {})
        first_step = first_chart.get('steps', [{}])[0] if first_chart.get('steps') else {}
        if 'time' in first_step:
            return True, len(existing)
    return False, 0

already_done, count = check_already_processed()
print(f"Already processed: {already_done} ({count} files found)")

Already processed: True (222 files found)


In [3]:
def normalize(path: Path, group: str) -> dict | None:
    with open(path, 'r') as file:
        data = json.load(file)

    music_path = data.get('music_fp', '')
    if music_path:
        song = Path(music_path).parent.name
    else:
        song = data.get('title', path.stem)

    output = {
        'dataset': group,
        'song_name': song,
        'charts': {}
    }

    seen = set()
    for chart in data.get('charts', []):
        difficulty = chart.get('difficulty_coarse', 'Unknown')
        if difficulty not in ALLOWED_DIFFICULTIES or difficulty in seen:
            continue
        seen.add(difficulty)

        steps = []
        for note in chart.get('notes', []):
            if len(note) >= 4:
                time_seconds = float(note[2])
                step = note[3]
                steps.append({'time': time_seconds, 'step': step})

        if steps:
            output['charts'][difficulty] = {
                'difficulty': difficulty,
                'steps': steps,
                'num_steps': len(steps)
            }

    return output if output['charts'] else None

In [4]:
if already_done:
    print(f"Skipping: {count} normalized human charts already exist in {OUTPUT_DIR}")
else:
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    for group in GROUPS:
        input_folder = UNPROCESSED_DIR / group
        output_folder = OUTPUT_DIR / group
        output_folder.mkdir(exist_ok=True)

        if not input_folder.exists():
            print(f"Skipping {group} - directory not found")
            continue

        files = list(input_folder.glob('**/*.json'))
        print(f"Processing {len(files)} files from {group}...")

        for path in files:
            chart_object = normalize(path, group)
            if chart_object:
                file_out = output_folder / f"{chart_object['song_name']}_human.json"
                with open(file_out, 'w') as write_file:
                    json.dump(chart_object, write_file, indent=2)

    total = sum(1 for _ in OUTPUT_DIR.glob('**/*_human.json'))
    print(f"Done! Created {total} normalized human chart files")

Skipping: 222 normalized human charts already exist in /Users/enscribe/Repositories/School/cse158-a2/data/real


In [5]:
sample_files = list(OUTPUT_DIR.glob('**/*_human.json'))[:3]
for f in sample_files:
    with open(f, 'r') as file:
        data = json.load(file)
    print(f"\n{f.name}:")
    print(f"  Song: {data['song_name']}")
    print(f"  Dataset: {data['dataset']}")
    print(f"  Difficulties: {list(data['charts'].keys())}")
    for diff, chart in data['charts'].items():
        print(f"    {diff}: {chart['num_steps']} steps")


Why Me_human.json:
  Song: Why Me
  Dataset: itg
  Difficulties: ['Easy', 'Medium', 'Hard', 'Challenge', 'Beginner']
    Easy: 131 steps
    Medium: 186 steps
    Hard: 288 steps
    Challenge: 423 steps
    Beginner: 56 steps

Soapy Bubble_human.json:
  Song: Soapy Bubble
  Dataset: itg
  Difficulties: ['Hard', 'Medium', 'Easy', 'Beginner', 'Challenge']
    Hard: 343 steps
    Medium: 204 steps
    Easy: 140 steps
    Beginner: 71 steps
    Challenge: 596 steps

Know Your Enemy_human.json:
  Song: Know Your Enemy
  Dataset: itg
  Difficulties: ['Easy', 'Medium', 'Hard', 'Challenge', 'Beginner']
    Easy: 165 steps
    Medium: 247 steps
    Hard: 402 steps
    Challenge: 585 steps
    Beginner: 47 steps
